# Reducing Stigma Toward People with Opioid Use Disorder Among Primary Care Clinicians

#### J M Maxwell - Data Science, Sr. Analyst - CTDS

In this notebook, we utilize data from the study “Reducing Stigma Toward People with Opioid Use Disorder Among Primary Care Clinicians” to recreate a number of the images and figures as seen in the paper by Hooker et al. titled *A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians* [(PMID: 36774521)](https://pubmed.ncbi.nlm.nih.gov/36774521/). The data for this study are archived in NIDA Data Share. 

The purpose of this notebook is to demonstrate how data accessed through the HEAL Data Platform can be analyzed in a HEAL workspace. Due to possible minor differences in selection/filtering of observations and handling of missing data, the tables and figures in this notebook may slightly vary from the results published in the paper.

The work here was conducted without direct involvement by Hooker et al., and does not represent study conclusions or recommendations on behalf of the NIH HEAL Initiative, The Center For Translational Data Science, or The University of Chicago.


## Introduction

There is an ongoing, dramatic rise in the number of opioid related overdose deaths in the United States, however only a small portion of patients who are diagnosed with Opioid Use Disorders (OUD) seek treatment and recieve medications for OUD (MOUD). MOUDs, like buprenorphine, can be effective forms of treatment for OUD patients, but only a portion of clinicians are waivered to, and in actuality do, prescribe buprenophine. A potential barrier to care for OUD patients may be the stigma held by primary care clinicians (PCCs) towards people with OUD. There are a limited number of interventions available for reducing OUD stigma among PCCs and a need for analytical support for the efficacy of these interventions. This study examines whether PCCs' stigma towards people with OUD may be reduced and PCCs' intentions to treat OUD patients increased through online training interventions.

#### Import Python Libraries

In [2]:
!pip install matplotlib -q
!pip install scikit_learn -q

import shutil
import os
import zipfile
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import sklearn
from sklearn import metrics

from IPython.display import Markdown, Image, display

folder_path = 'img/PCC_OUD_Stigma'
if os.path.exists(folder_path) and os.path.isdir(folder_path):
    shutil.rmtree(folder_path)
    os.makedirs(folder_path)
    print(f"A new folder - '{folder_path}' - has been created.")
else:
    os.makedirs(folder_path)
    print(f"A new folder - '{folder_path}' - has been created.")

A new folder - 'img/PCC_OUD_Stigma' - has been created.


## Ingest And Prepare Data

In this section we ingest and prepare the data. We're particularly interested in the data fields related to the PCCs' attitude towards people with OUD, their attitudes towards treating people with OUD, and their demographic information. 

In [3]:
!gen3 drs-pull object dg.H34L/a6c7fc23-3534-401c-88ca-9fbbb495dcf4

{"succeeded": ["dg.H34L/a6c7fc23-3534-401c-88ca-9fbbb495dcf4"], "failed": []}


In [4]:
with zipfile.ZipFile('ascii-crf-data-files_nida-ctn-0095a2 v1-1.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [5]:
df = pd.read_csv('stigma_supp.csv')
col_names = {'DDBS': 'Overall Stigma', 'DDBS_DIFF': 'Difference', 'DDBS_DISDAIN': 'Disdain', 'DDBS_BLAME2': 'Blame', 
              'INTEND_WAIVER': 'Intentions to get waivered', 'INTEND_PRESCRIBE': 'Intentions to prescribe buprenorphine', 
              'WILLWORK': 'Willingness to work with OUD', 'TX_EFFECTIVE': 'Perceived OUD treatment effectiveness', 
              'TX_ADHERENCE': 'Perceived OUD treatment adherence', 'PCC_AGE': 'Age', 'PCC_GENDER': 'Gender', 'PCC_RACE': 'Race', 
              'PCC_HISPANIC': 'Ethnicity', 'STIGMA_GROUP': 'Stigma Group', 'PCC_WAIVERED': 'Waivered to prescribe buprenorphine', 'PCC_MD': 'Degree'}
df.rename(columns=col_names, inplace=True)
df.drop(['STIG_CLIN_ID', 'HLTH_SYS_ID', 'PCC_ID', 'COMPLETE_TRAINING', 'COMPLETE_SURVEY'], axis=1, inplace=True)
df.Gender = df.Gender.fillna(7.0)
df.Ethnicity = df.Ethnicity.fillna(7.0)
df['Race'] = df.Race.map({'5 White': 'White', '9 Prefer not to answer': 'Prefer not to answer', '2 Asian': 'Asian', '3 Black or African American': 'Black or African American', 
                          '7 Multiple selected': 'Multiple selected', '6 Some other race': 'Some other race'})
df['Gender'] = df.Gender.map({2.0: 'Female', 1.0: 'Male', 6.0: 'Not Listed', 7.0: 'Prefer not to answer'})
df['Ethnicity'] = df.Ethnicity.map({0.0: 'Not Hispanic or Latino', 1.0: 'Hispanic or Latino',  7.0: 'Missing'})
df['Waivered to prescribe buprenorphine'] = df['Waivered to prescribe buprenorphine'].map({0: 'False', 1: 'True'})
df['Degree'] = df.Degree.map({1: 'MD/DO', 0: 'PA/NP'})
df = df[['Stigma Group', 'Overall Stigma', 'Difference', 'Disdain', 'Blame',  'Intentions to get waivered', 'Intentions to prescribe buprenorphine', 'Willingness to work with OUD', 
         'Perceived OUD treatment effectiveness', 'Perceived OUD treatment adherence', 'Waivered to prescribe buprenorphine', 'Gender', 'Ethnicity', 'Race', 'Degree']]

Markdown(df.head().to_markdown())

|    | Stigma Group   |   Overall Stigma |   Difference |   Disdain |   Blame |   Intentions to get waivered |   Intentions to prescribe buprenorphine |   Willingness to work with OUD |   Perceived OUD treatment effectiveness |   Perceived OUD treatment adherence | Waivered to prescribe buprenorphine   | Gender   | Ethnicity              | Race   | Degree   |
|---:|:---------------|-----------------:|-------------:|----------:|--------:|-----------------------------:|----------------------------------------:|-------------------------------:|----------------------------------------:|------------------------------------:|:--------------------------------------|:---------|:-----------------------|:-------|:---------|
|  0 | STIG_CTRL      |             5.75 |         5.33 |      6.33 |     5.5 |                            1 |                                       3 |                           2    |                                       2 |                                   2 | False                                 | Male     | Not Hispanic or Latino | White  | MD/DO    |
|  1 | STIG_INT       |             4.5  |         3.33 |      5.33 |     5   |                            2 |                                       2 |                           3    |                                       3 |                                   3 | False                                 | Male     | Not Hispanic or Latino | White  | MD/DO    |
|  2 | STIG_CTRL      |             4.88 |         4.33 |      5.33 |     5   |                            2 |                                       3 |                           3    |                                       3 |                                   2 | False                                 | Male     | Not Hispanic or Latino | White  | MD/DO    |
|  3 | STIG_INT       |             4    |         3.67 |      4.67 |     3.5 |                            2 |                                       2 |                           2.67 |                                       2 |                                   3 | False                                 | Female   | Not Hispanic or Latino | White  | MD/DO    |
|  4 | STIG_INT       |             4.25 |         4.33 |      5    |     3   |                            3 |                                       3 |                           2.67 |                                       3 |                                   2 | False                                 | Female   | Not Hispanic or Latino | White  | PA/NP    |

## About The Study

The study used by Hooker et. al. for their research was conducted by a nonprofit healthcare organization, HealthPartners, servicing Minnesota and Wisconsin. PCCs who are part of the 15 HealPartners clinics that have access to a CDS tool for identifying and treating people with OUD were invited to participate in the study. The study was a randomized, controlled trial where PCCs recieved one of two trainings: a stigma reduction training or an attention-control training. Following training, the PCCs were assesed at 6 months, 3 months and immediately following completion of the training for their stigma against people with OUD, their intention to get waivered, and their intention to prescribe buprenorphine. Of the 162 PCCs who were eligible, 85 completed both the training and the immediate post training survey (n=46 stigma training and n=39 attention-control training). 

The stigma reduction training was a series of videos recounting OUD patient narratives, examples of using non-stigmatizing language, and explanantions on how to use a Clincal Decision Support (CDS) tool designed to enable PCCs to screen, diagnose, and treat patients with OUD. The attention-control training controlled for contact, attention, and extra CDS tool training, but did not include the OUD patient narratives or examples of non-stigmatized language. 

The outcome measures which were used in this analysis were assessed through a survey immediately following training. The relevant primary and secondary outcome measurements were: 
- *OUD Stigma* - measured using the *Blame*, *Disdain*, and *Difference* scales and the composite *Overall Stigma* scale.
- *Intentions to get waivered or prescribe buprenorphine* - measured on a 1-5 scale the likelihood to get waivered to prescribe buprenorphine or to prescribe buprenorphine should a waiver no longer be required. 
- *Willingness to work with people with OUD* - measured as the average of 3 questionnaire items (each scaled 1-5) from the Drug Problems Perceptions Questionnaire, where a higher score indicates a greater overall willingness to work with patients with OUD.
- *Opioid treatment outcome expectations* - measured on a scale 1-4 on PCCs expecations for the effectiveness and likelihood of patient adherence to OUD treatment.

## Demographic Summary Statistics of PCCs who completed stigma or attention control training 

In [6]:
features = ['Gender', 'Ethnicity', 'Race', 'Waivered to prescribe buprenorphine', 'Degree']
all_subj = []
print_df = pd.DataFrame()

for feat in features:
    print_df1 = pd.DataFrame({'Category':  f'{feat} - ' + df[feat].value_counts().index, 'All N=85': (100*df[feat].value_counts()/len(df)).values.round(2)} )
    print_df2 = pd.DataFrame({'Category': f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).index, 'Stigma reduction n = 46': (100*df[df['Stigma Group'] == 'STIG_INT'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_INT'])).values.round(2)} )
    print_df3 = pd.DataFrame({'Category':  f'{feat} - ' + (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).index, 'Attention-control n = 39':  (100*df[df['Stigma Group'] == 'STIG_CTRL'][feat].value_counts()/len(df[df['Stigma Group'] == 'STIG_CTRL'])).values.round(2)} )
    
    print_df1 = print_df1.merge(print_df2, how='left', on='Category')
    print_df1 = print_df1.merge(print_df3, how='left', on='Category')
    print_df1.fillna(0, inplace=True)
    print_df1[['All N=85', 'Stigma reduction n = 46', 'Attention-control n = 39']] = print_df1[['All N=85', 'Stigma reduction n = 46', 'Attention-control n = 39']].astype(str)
    print_df1['All N=85'] = print_df1['All N=85'].apply( lambda x : str(x) + '%')
    print_df1['Stigma reduction n = 46'] = print_df1['Stigma reduction n = 46'].apply( lambda x : str(x) + '%')
    print_df1['Attention-control n = 39'] = print_df1['Attention-control n = 39'].apply( lambda x : str(x) + '%')
    print_df = pd.concat([print_df, print_df1])
    
    if feat != 'Degree':
        print_df = pd.concat([print_df,  pd.DataFrame({'Category': ['-'], 'All N=85': ['-'], 'Stigma reduction n = 46': ['-'], 'Attention-control n = 39': ['-'] })])
        
Markdown(print_df.to_markdown())

|    | Category                                    | All N=85   | Stigma reduction n = 46   | Attention-control n = 39   |
|---:|:--------------------------------------------|:-----------|:--------------------------|:---------------------------|
|  0 | Gender - Female                             | 57.65%     | 58.7%                     | 56.41%                     |
|  1 | Gender - Male                               | 36.47%     | 36.96%                    | 35.9%                      |
|  2 | Gender - Prefer not to answer               | 4.71%      | 2.17%                     | 7.69%                      |
|  3 | Gender - Not Listed                         | 1.18%      | 2.17%                     | 0.0%                       |
|  0 | -                                           | -          | -                         | -                          |
|  0 | Ethnicity - Not Hispanic or Latino          | 94.12%     | 97.83%                    | 89.74%                     |
|  1 | Ethnicity - Hispanic or Latino              | 3.53%      | 2.17%                     | 5.13%                      |
|  2 | Ethnicity - Missing                         | 2.35%      | 0.0%                      | 5.13%                      |
|  0 | -                                           | -          | -                         | -                          |
|  0 | Race - White                                | 67.06%     | 63.04%                    | 71.79%                     |
|  1 | Race - Prefer not to answer                 | 14.12%     | 15.22%                    | 12.82%                     |
|  2 | Race - Asian                                | 9.41%      | 13.04%                    | 5.13%                      |
|  3 | Race - Black or African American            | 7.06%      | 6.52%                     | 7.69%                      |
|  4 | Race - Multiple selected                    | 1.18%      | 0.0%                      | 2.56%                      |
|  5 | Race - Some other race                      | 1.18%      | 2.17%                     | 0.0%                       |
|  0 | -                                           | -          | -                         | -                          |
|  0 | Waivered to prescribe buprenorphine - False | 90.59%     | 93.48%                    | 87.18%                     |
|  1 | Waivered to prescribe buprenorphine - True  | 9.41%      | 6.52%                     | 12.82%                     |
|  0 | -                                           | -          | -                         | -                          |
|  0 | Degree - MD/DO                              | 64.71%     | 65.22%                    | 64.1%                      |
|  1 | Degree - PA/NP                              | 35.29%     | 34.78%                    | 35.9%                      |

In this first table we have a breakdown of the demographic characteristics of the participating PCCs split amongst the two training formats. There are no significant differences between the stigma and attention-control training groups in their demographics, their medical degree type, or their pre-existing waiver to prescribe burprenorphine. 

## Stigma Reduction and Attention-Control Training on Self-Reported Stigma and Intentions to Treat People 

In [7]:
stig_int = []
stig_ctrl = []
t_vals = []
p_vals = []
d_vals = []
features = ['Overall Stigma', 'Difference', 'Disdain', 'Blame',  'Intentions to get waivered', 
            'Intentions to prescribe buprenorphine', 'Willingness to work with OUD', 'Perceived OUD treatment effectiveness', 'Perceived OUD treatment adherence']
def cohen_d(x,y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    return (np.mean(x) - np.mean(y)) / np.sqrt(((nx-1)*np.std(x, ddof=1) ** 2 + (ny-1)*np.std(y, ddof=1) ** 2) / dof)

for feat in features:
    x = df[df['Stigma Group'] == 'STIG_INT'][feat].dropna()
    y = df[df['Stigma Group'] == 'STIG_CTRL'][feat].dropna()
    
    # Perform the t-test
    t_statistic, p_value = scipy.stats.ttest_ind(x, y)
    d = cohen_d(x, y).round(2)
    
    stig_int.append(f'{x.mean().round(1)} ({x.std().round(1)})')
    stig_ctrl.append(f'{y.mean().round(1)} ({y.std().round(1)})')
    t_vals.append(t_statistic)
    p_vals.append(p_value)
    d_vals.append(d)

df1 = pd.DataFrame({"Feature": features, 
                    "Stigma Reduction Mean (SD)": stig_int,
                    "Attention Control Mean (SD)": stig_ctrl,
                    "t-statistic": np.round(t_vals, 3),
                    "p-value": np.round(p_vals, 3),
                    "Cohen's d-value": d_vals})
Markdown(df1.to_markdown())

|    | Feature                               | Stigma Reduction Mean (SD)   | Attention Control Mean (SD)   |   t-statistic |   p-value |   Cohen's d-value |
|---:|:--------------------------------------|:-----------------------------|:------------------------------|--------------:|----------:|------------------:|
|  0 | Overall Stigma                        | 4.1 (1.3)                    | 4.2 (1.2)                     |        -0.481 |     0.632 |             -0.1  |
|  1 | Difference                            | 3.4 (1.7)                    | 3.1 (1.7)                     |         0.741 |     0.461 |              0.16 |
|  2 | Disdain                               | 4.7 (1.4)                    | 4.9 (1.4)                     |        -0.859 |     0.393 |             -0.19 |
|  3 | Blame                                 | 4.4 (1.6)                    | 4.8 (1.6)                     |        -1.286 |     0.202 |             -0.28 |
|  4 | Intentions to get waivered            | 2.3 (0.7)                    | 2.1 (0.8)                     |         1.113 |     0.269 |              0.26 |
|  5 | Intentions to prescribe buprenorphine | 3.2 (1.0)                    | 3.0 (0.9)                     |         0.899 |     0.372 |              0.21 |
|  6 | Willingness to work with OUD          | 3.0 (0.7)                    | 3.1 (0.9)                     |        -0.828 |     0.41  |             -0.18 |
|  7 | Perceived OUD treatment effectiveness | 2.6 (0.8)                    | 2.7 (0.7)                     |        -0.745 |     0.459 |             -0.16 |
|  8 | Perceived OUD treatment adherence     | 2.5 (0.6)                    | 2.4 (0.6)                     |         0.15  |     0.881 |              0.03 |

In the second table we show a number of descriptive statistics for measuring the difference amongst the two training groups in self-reported stigma (Overall Stigma, Difference, Disdain, and Blame), intentions for getting waivered or to prescribe buprenophine if waivered, and perceptions of people with OUD and OUD treatment. The mean and standard deviations for both training groups, as well as, the t-statistic, corresponding p-value, and Cohen's d-value all demonstrate that there are minimal or no significant differences between the two training groups across all measured training outcomes. 

## Correlations among Self-Reported Sitgma and Intentions to Treat People

In [8]:
corr_matrix = df[features].corr().round(2)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_matrix.columns = ['1', '2', '3', '4', '5', '6', '7', '8', '9']

Markdown(corr_matrix.mask(mask).to_markdown())

|                                       |      1 |      2 |      3 |      4 |      5 |      6 |      7 |      8 |   9 |
|:--------------------------------------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|-------:|----:|
| Overall Stigma                        | nan    | nan    | nan    | nan    | nan    | nan    | nan    | nan    | nan |
| Difference                            |   0.84 | nan    | nan    | nan    | nan    | nan    | nan    | nan    | nan |
| Disdain                               |   0.79 |   0.47 | nan    | nan    | nan    | nan    | nan    | nan    | nan |
| Blame                                 |   0.64 |   0.31 |   0.33 | nan    | nan    | nan    | nan    | nan    | nan |
| Intentions to get waivered            |  -0.25 |  -0.06 |  -0.23 |  -0.35 | nan    | nan    | nan    | nan    | nan |
| Intentions to prescribe buprenorphine |  -0.25 |  -0.11 |  -0.18 |  -0.35 |   0.61 | nan    | nan    | nan    | nan |
| Willingness to work with OUD          |  -0.4  |  -0.34 |  -0.29 |  -0.34 |   0.42 |   0.46 | nan    | nan    | nan |
| Perceived OUD treatment effectiveness |  -0.32 |  -0.19 |  -0.26 |  -0.38 |   0.21 |   0.29 |   0.46 | nan    | nan |
| Perceived OUD treatment adherence     |  -0.39 |  -0.28 |  -0.31 |  -0.35 |   0.17 |   0.22 |   0.36 |   0.62 | nan |

In the final table above, we can see how self-reported stigma, intentions for getting waivered or to prescribe buprenophine if waivered, and perceptions of people with OUD and OUD treatment are related. We use Pearson's correlation coefficients to measure the relation between the measured training outcomes. 

Unsurprisingly, the components for measuring stigma are strongly, positively correlated. We can also see moderate or high positive correlations between Perceived OUD treatment adherence and treatment effectiveness, between PCCs' intentions to get waivered and to prescribe buprenorphine, and between PCCs' willingness to work with OUD patients and the PCCs intentions to prescribe buprenorphine and their perceived OUD treatment effectiveness/adherence. 

There is also a slight or moderate negative correlation between PCCs' self-reported stigma and their intensions to work with OUD patients, their percieved OUD treatment effectiveness/adherence, and their intention to get waivered and prescribe buprenorphine. 

## Conclusions

The study demonstrated there was little relation between the provided stigma reduction training and PCCs' stigma, future intentions, willingness to work with OUD patients, or PCCs' perception of treatment effectiveness or patient treatment adherence. As was expected, there was a relation amongst PCCs' established stigma and their willingness to work with OUD patients and their perceptions of OUD treatment effectiveness and adherence. 

Since greater established PCC stigma is shown to inversely relate to PCCs' willingness to treat people with OUD, intentions to prescribe buprenophine, and perceptions regarding treatment effectiveness and patient adherence to treatment, it is important to find more effective intervention methods for reducing stigma. 

## References

#### Paper
Hooker, Stephanie A et al. “A randomized controlled trial of an intervention to reduce stigma toward people with opioid use disorder among primary care clinicians.” Addiction science & clinical practice vol. 18,1 10. 11 Feb. 2023, doi:10.1186/s13722-023-00366-1

#### Study
Hooker, S., Rossum, R., Crain, L., Bart, G. "Reducing Stigma toward People with Opioid Use Disorder among Primary Care Clinicians." NIDA Data Share. May 2024, NIDA-CTN-0095A2

#### HEAL Data Platform
Hooker, Stephanie A.& Bart, Gavin & Rossom, Rebecca  (2025). Reducing Stigma Toward People with Opioid Use Disorder Among Primary Care Clinicians 2. HEAL Data Platform. Study Record. 10.60490/HDP01287